In [ ]:
# ===========================================================================
# ACOUSTIC FEATURE ANALYSIS - interactive driver
#
# All extraction/plotting logic lives in src/; this notebook only calls it
# and stores results, matching notebooks/01's convention. Split out of the
# old notebooks/01_data_pipeline.ipynb (EDA) and notebooks/03_praat_analysis.ipynb
# (Praat extraction/statistics) into one feature-analysis notebook, since both
# are exploratory acoustic-feature work, not pipeline construction.
#
#   src/preprocessing.py   MFCC extraction (13 coeff + delta + delta-delta)
#   src/praat.py            per-feature-group extraction (F0, jitter, shimmer,
#                            HNR, CPPS, formants, intensity, speech-rate/pause/
#                            voice-break proxies) + extract_praat_features_batch()
#   src/visualization.py    EDA figures, Praat box plots/summary, feature
#                            correlation heatmap
#
# Features are extracted from the ORIGINAL audio, not the VAD-trimmed/
# zero-padded window the Deep/Acoustic pathways train on - jitter, shimmer,
# HNR, and formants are only meaningful on natural speech. See ROADMAP.md
# Phase 4 for the full plan and src/praat.py's module docstring for the
# speech-rate/pause-duration caveat (UA-Speech utterances are single
# isolated words, not continuous speech).
# ===========================================================================

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.console import print_header, print_kv
from src.training.data import load_manifest

config.ensure_directories()

df_m6 = load_manifest()

print_header("Acoustic Feature Analysis")
print_kv("Manifest", config.MANIFEST_PATH)
print_kv("Utterances", len(df_m6))

In [ ]:
# STAGE 1 - Dataset visualization / EDA, on every scanned audio file (all mic
# channels, before the M6 filter) - moved from notebooks/01_data_pipeline.ipynb
# Stage 3. Re-scans the extracted audio (src.scanning.scan_audio_files, the
# same call notebooks/01 Stage 2 makes) rather than depending on 01's
# in-memory df_audio or the M6-only manifest, so this cell reproduces the
# same dashboard whether or not notebook 01's kernel is still open. Figures
# render inline and are also saved to outputs/figures/ at 300 dpi.
from src.scanning import scan_audio_files
from src.visualization import run_eda

df_audio = scan_audio_files()
run_eda(df_audio, show=True)

In [ ]:
# STAGE 2 - MFCC extraction, standalone. UASpeechDataset (src/dataset.py) calls
# src.preprocessing.extract_mfcc_features internally for every training batch;
# this cell runs the identical two-line call on one utterance so the 39-dim
# (13 MFCC + delta + delta-delta) representation the Acoustic Pathway trains
# on is visible on its own, not just implicitly inside the dataset.
import matplotlib.pyplot as plt

from src.preprocessing import build_mfcc_transform, extract_mfcc_features, load_and_preprocess

sample_row = df_m6.iloc[0]
mfcc_transform = build_mfcc_transform()

waveform = load_and_preprocess(sample_row["Filepath"])
mfcc = extract_mfcc_features(waveform, mfcc_transform)[0].numpy()

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(mfcc, aspect="auto", origin="lower", cmap="viridis")
ax.set_title(f"MFCC + delta + delta-delta — {sample_row['Filename']}")
ax.set_xlabel("Frame")
ax.set_ylabel("Coefficient (39 = 13 MFCC + 13 delta + 13 delta-delta)")
fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()
fig.savefig(config.FIGURE_DIR / "mfcc_sample.png", dpi=300, bbox_inches="tight")
plt.show()

print_header("MFCC Extraction")
print_kv("Sample file", sample_row["Filename"])
print_kv("MFCC tensor shape", mfcc.shape)
print_kv("Mean / std", f"{mfcc.mean():.4f} / {mfcc.std():.4f}")

In [ ]:
# STAGE 3 - Extract all FEATURE_COLUMNS Praat features for every utterance in the manifest:
#   pitch        f0 mean/max/min/std/range        (std+range = monopitch)
#   perturbation jitter x4, shimmer x4            (vocal-fold instability)
#   noise        hnr mean/std/min                 (breathiness, roughness)
#   voice qual cpps                             (breathy/dysphonic voice quality)
#   articulation f1/f2/f3 mean+std, f2_f1_ratio   (vowel-space centralization)
#   loudness     intensity mean/max/min/std       (loudness control)
#   rhythm       speech_rate, pause_duration, voice_breaks
#
# Cached to outputs/praat_features.csv - safe to re-run this cell, later runs
# load the cache instead of recomputing ~21k files (~25-30 minutes uncached).
from src.praat import extract_praat_features_batch

praat_features = extract_praat_features_batch(df_m6, cache_path=config.PRAAT_FEATURES_PATH)
praat_features.head()

In [ ]:
# STAGE 4 - Acoustic feature statistics: compare Healthy vs Very Low vs Low vs
# Mid vs High severity groups - box-plot grid for all FEATURE_COLUMNS features,
# saved to outputs/figures/, plus a group-means +/- std table saved to
# outputs/metrics/.
from src.visualization import plot_praat_feature_comparison, build_praat_group_summary

figure_path = plot_praat_feature_comparison(praat_features, show=True)
group_summary = build_praat_group_summary(praat_features)

summary_path = config.METRICS_DIR / "praat_severity_group_summary.csv"
group_summary.to_csv(summary_path)

print_header("Severity Group Comparison")
print_kv("Figure", figure_path)
print_kv("Group summary table", summary_path)
group_summary

In [ ]:
# STAGE 5 - Feature correlation analysis. Which Praat features move together?
# Jitter/shimmer sub-measures and the three formant means are expected to
# correlate strongly since they're different formulas over the same
# underlying signal - this is what tells a reader which features carry
# genuinely independent evidence before Phase 6's Praat pathway treats all
# FEATURE_COLUMNS as independent inputs.
from src.visualization import plot_feature_correlation

correlation_figure = plot_feature_correlation(praat_features, show=True)
print_header("Feature Correlation")
print_kv("Figure", correlation_figure)

In [ ]:
# STAGE 6 - Which of those group differences are actually real?
#
# The box plots above are descriptive; this is the test. Kruskal-Wallis H per
# feature across the five groups (non-parametric, because jitter/shimmer and the
# pause/voice-break proxies are bounded and heavily skewed, so ANOVA's normality
# assumption does not hold), Bonferroni-corrected across the FEATURE_COLUMNS features so that
# testing them all at once does not manufacture significance.
#
# The significant rows are the features the discussion section can legitimately
# claim separate severity levels - and they are the ones worth reading first in
# notebooks/05_error_analysis.ipynb and Phase 6's Praat fusion pathway.
from src.praat import praat_group_significance

significance = praat_group_significance(praat_features)

significance_path = config.METRICS_DIR / "praat_significance.csv"
significance.to_csv(significance_path, index=False)

n_sig = int(significance["significant"].sum())

print_header("Kruskal-Wallis Severity Group Significance")
print_kv("Significance table", significance_path)
print_kv("Significant features (p_adj < 0.05)", f"{n_sig} / {len(significance)}")
print_kv("Strongest separator", significance.iloc[0]["feature"])
significance